In [ ]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tokenizers import ByteLevelBPETokenizer
import os
from transformers import RobertaTokenizerFast
import torch
import torch.nn as nn
import math
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

In [ ]:
from google.colab import files
#from google.colab import files
uploaded = files.upload()

# Assume file name is humor_corpus.txt
file_path = list(uploaded.keys())[0]
print("Uploaded:", file_path)


Saving my_dataset2.tsv to my_dataset2 (1).tsv
Uploaded: my_dataset2 (1).tsv


In [ ]:


# Load TSV file
df = pd.read_csv("my_dataset2.tsv", sep="\t")  # important: sep="\t"
print(df.head())


       input_text                                        target_text
0      moon llama  The llama moved to the moon for better grazing...
1  angry broccoli  Broccoli tried yelling at kids to eat vegetabl...
2  lonely penguin  The penguin posted on social media: “Looking f...
3   dizzy toaster  My toaster spun in circles claiming it was reh...
4  robot squirrel  A robot squirrel hacked my bird feeder for “re...


In [ ]:
tokenizer = ByteLevelBPETokenizer()

tokenizer.train(
    files=file_path,
    vocab_size=30000,
    min_frequency=2,
    special_tokens=[
        "<s>", "<pad>", "</s>", "<unk>", "<mask>"
    ],
)

tokenizer.save_model("humor-tokenizer")

['humor-tokenizer/vocab.json', 'humor-tokenizer/merges.txt']

In [ ]:
hf_tokenizer = RobertaTokenizerFast.from_pretrained(
    "humor-tokenizer",
    unk_token="<unk>",
    pad_token="<pad>",
    cls_token="<s>",
    sep_token="</s>",
    mask_token="<mask>",
)

hf_tokenizer.save_pretrained("humor-tokenizer")

('humor-tokenizer/tokenizer_config.json',
 'humor-tokenizer/special_tokens_map.json',
 'humor-tokenizer/vocab.json',
 'humor-tokenizer/merges.txt',
 'humor-tokenizer/added_tokens.json',
 'humor-tokenizer/tokenizer.json')

In [ ]:

tokenizer = RobertaTokenizerFast.from_pretrained("humor-tokenizer")


vocab_size = tokenizer.vocab_size
print("Vocab size:", vocab_size)

Vocab size: 2661


In [ ]:
class HumorDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.examples = []
        self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

        for _, row in df.iterrows():
            # Format the text
            text = f"<s>input: {row['input_text']}\njoke: [{row['target_text']}]</s>"
            ids = tokenizer.encode(text)

            # Truncate or Pad
            if len(ids) > max_len:
                ids = ids[:max_len]
            else:
                ids += [self.pad_token_id] * (max_len - len(ids))

            self.examples.append(ids)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ids = torch.tensor(self.examples[idx], dtype=torch.long)
        x = ids[:-1]
        y = ids[1:].clone()


        padding_mask = (x == self.pad_token_id)

        y[y == self.pad_token_id] = -100

        return x, y, padding_mask

In [ ]:
class GeneralTextDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_len=64):
        self.tokenizer = tokenizer


        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()


        print("Tokenizing general text... (this may take a moment)")
        full_ids = tokenizer.encode(text, add_special_tokens=False)

        self.examples = []
        self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
        step = max_len
        for i in range(0, len(full_ids) - max_len, step):
            chunk = full_ids[i : i + max_len]
            self.examples.append(chunk)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ids = torch.tensor(self.examples[idx], dtype=torch.long)
        x = ids[:-1]
        y = ids[1:].clone()


        padding_mask = (x == self.pad_token_id)
        y[y == self.pad_token_id] = -100

        return x, y, padding_mask


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, max_len=512):
        super().__init__()

        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.register_buffer("mask", torch.tril(torch.ones(max_len, max_len)))

    def forward(self, x, key_padding_mask=None):
        B, T, C = x.size()
        causal_mask = self.mask[:T, :T] == 0


        out, _ = self.attn(x, x, x,
                           attn_mask=causal_mask,
                           key_padding_mask=key_padding_mask)
        return out

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_dim, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_hidden_dim),
            nn.GELU(),
            nn.Linear(ff_hidden_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x, key_padding_mask=None):
        x = x + self.attn(self.ln1(x), key_padding_mask=key_padding_mask)
        x = x + self.ff(self.ln2(x))
        return x

In [ ]:
class MiniTransformerLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, num_heads=4, num_layers=4, ff_hidden_dim=512, max_len=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Parameter(torch.zeros(1, max_len, embed_dim))
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_hidden_dim)
            for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, idx, key_padding_mask=None):
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb[:, :T, :]
        for layer in self.layers:
            x = layer(x, key_padding_mask)
        x = self.ln_f(x)
        logits = self.head(x)
        return logits

In [ ]:
dataset = HumorDataset(df, tokenizer, max_len=128)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

In [ ]:
def train_transformer(model, train_loader, epochs=50, lr=3e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(ignore_index=-100) # Ignore padding

    for epoch in range(epochs):
        total_loss = 0
        # IMPORTANT: Unpack x, y, AND mask here
        for x, y, mask in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            x, y, mask = x.to(device), y.to(device), mask.to(device)

            optimizer.zero_grad()
            logits = model(x, key_padding_mask=mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            loss.backward()

            # Clip gradients to prevent crashes
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        # Print every 10 epochs
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

    return model

In [ ]:
def generate_joke(model, tokenizer, input_text, temperature=1.2):
    model.eval()
    device = next(model.parameters()).device

    # 1. Create the prompt exactly how the model expects it
    prompt = f"<s>input: {input_text}\njoke: ["

    # 2. Encode WITHOUT adding extra special tokens
    idx = torch.tensor([tokenizer.encode(prompt, add_special_tokens=False)], dtype=torch.long).to(device)

    # 3. Generate
    with torch.no_grad():
        for _ in range(50): # Max length
            logits = model(idx)
            logits = logits[:, -1, :] / temperature
            probs = torch.nn.functional.softmax(logits, dim=-1)

            # Sample a token
            next_token = torch.multinomial(probs, 1)

            # Stop if pad or eos
            if next_token.item() == tokenizer.pad_token_id or next_token.item() == tokenizer.eos_token_id:
                break

            idx = torch.cat((idx, next_token), dim=1)

            # Stop if we see the closing bracket ']'
            if "]" in tokenizer.decode(next_token[0]):
                break

    return tokenizer.decode(idx[0], skip_special_tokens=True)

In [ ]:
import requests

url = "https://www.gutenberg.org/files/2600/2600-0.txt"

print("Downloading 'War and Peace'... (This is a large file)")
r = requests.get(url)

full_text = r.text
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

start_idx = full_text.find(start_marker)
end_idx = full_text.find(end_marker)

if start_idx != -1 and end_idx != -1:
    start_idx += len(start_marker)
    book_content = full_text[start_idx:end_idx].strip()
else:
    book_content = full_text

# Save it
with open("general_english.txt", "w", encoding="utf-8") as f:
    f.write(book_content)

print(f"Success! Downloaded 'War and Peace'. Size: {len(book_content)/1024/1024:.2f} MB")

Success! Downloaded 'War and Peace'. Size: 3.12 MB


In [ ]:
general_dataset = GeneralTextDataset("general_english.txt", tokenizer, max_len=64)
general_loader = DataLoader(general_dataset, batch_size=16, shuffle=True)


model = MiniTransformerLM(vocab_size=tokenizer.vocab_size)


print("Phase 1: Learning English (Context)...")
model = train_transformer(model, general_loader, epochs=5, lr=5e-4)


general_dataset = GeneralTextDataset("stand-up.txt", tokenizer, max_len=64)
general_loader = DataLoader(general_dataset, batch_size=16, shuffle=True)
print("Phase 1.1: Learning English (Context)...")
model = train_transformer(model, general_loader, epochs=5, lr=5e-4)


torch.save(model.state_dict(), "pretrained_english_model.pt")
print("Phase 1 Complete. Model now knows English context.")

Tokenizing general text... (this may take a moment)
Phase 1: Learning English (Context)...


Tokenizing general text... (this may take a moment)
Phase 1.1: Learning English (Context)...


Phase 1 Complete. Model now knows English context.


In [ ]:
# --- RUN FINE-TUNING ---

# 1. Load the "Smart" Model
model = MiniTransformerLM(vocab_size=tokenizer.vocab_size)
model.load_state_dict(torch.load("pretrained_english_model.pt"))
print("Loaded Pre-trained English Model.")

# 2. Load your Joke Dataset (The small one)
df = pd.read_csv("my_dataset2.tsv", sep="\t")
joke_dataset = HumorDataset(df, tokenizer, max_len=64)
joke_loader = DataLoader(joke_dataset, batch_size=16, shuffle=True)

# 3. Fine-tune
# LOWER LR: 1e-4 or 5e-5 (Don't break the pre-training)
# FEWER EPOCHS: Stop before it memorizes (Loss ~1.0)
print("Phase 2: Learning to Joke...")
model = train_transformer(model, joke_loader, epochs=15, lr=5e-5) # Low LR!

# 4. Test
print("\n--- GENERATED JOKE ---")
print(generate_joke(model, tokenizer, "tired banana", temperature=0.8))

Loaded Pre-trained English Model.
Phase 2: Learning to Joke...


Epoch 10 | Loss: 0.6122



--- GENERATED JOKE ---
input: tired banana
joke: [The banana wore a top hat on its shell.]


In [ ]:
print("\n--- GENERATED JOKE ---")
print(generate_joke(model, tokenizer, "tired banana", temperature=0.8))


--- GENERATED JOKE ---
input: tired banana
joke: [The banana hummed pop songs at 2 a.m.]


In [ ]:
# 4. Test
print("\n--- GENERATED JOKE ---")
print(generate_joke(model, tokenizer, "sad tomato", temperature=0.5))




--- GENERATED JOKE ---
input: sad tomato
joke: [The tomato smiled while spinning messy webs.]


In [ ]:
# 4. Test
print("\n--- GENERATED JOKE ---")
print(generate_joke(model, tokenizer, "sleepy banana", temperature=0.5))


--- GENERATED JOKE ---
input: sleepy banana
joke: [The banana felt missing midnight.]
